# Candidato final: más épocas, menos weight decay y menos peso a las clases raras

Una sola corrida del pipeline completo con los dos modelos (`bidirectional_rnn` y `joint_conv_bilstm`)
y tres cambios respecto de las corridas existentes:

| | Referencia | Candidato | Por qué |
|---|---|---|---|
| Épocas | 5 | **10** | Los modelos subajustan: la mejor época era la 4 o la 5 y la loss de train seguía bajando. El pipeline se queda con la mejor época en test, así que entrenar de más no empeora el modelo guardado. |
| Weight decay de la BiRNN de puntuación (λ) | 1.0 | **0.1** | λ = 1 se eligió en `regularization_weight_decay.ipynb` con ~300 pasos. El decay de AdamW acumula `exp(−lr · λ · pasos)`: con ~62 mil instancias de train y 10 épocas son ~3100 pasos, así que λ = 0.1 mantiene el mismo decay total (λ = 1 achicaría los pesos un 95%). |
| `class_weight_power` | 0.5 | **0.25** | Las clases raras (`¿`, `,`, `?`, MiXta, MAYÚS) tenían recall mucho mayor que precisión: se predecían de más. |

Datos: hasta `1_000_000` instancias por fuente (~62 mil instancias de train), igual que la corrida `2026-11-16__21-24-10`.

Se compara contra el mejor resultado existente sobre **control** (test ya se usa para elegir la época).

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**.
Los resultados se guardan en Drive (`MyDrive/tp2-models`).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    # Los resultados van a Drive: la máquina de Colab se borra al cerrar la sesión
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
    !mkdir -p /content/drive/MyDrive/tp2-models
    !if [ ! -L models ]; then rm -rf models && ln -s /content/drive/MyDrive/tp2-models models; fi
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    %cd ../..

!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Configuración

In [ ]:
EPOCHS = 10
SIZE = 1_000_000  # máximo de instancias por fuente, igual que 21-24-10
PUNCTUATION_BIRNN_WEIGHT_DECAY = 0.1  # referencia: 1.0 (src/problem/punctuation/__init__.py)
CLASS_WEIGHT_POWER = 0.25  # referencia: 0.5 (src/constants.py)

# Pisamos los valores antes de importar el pipeline, sin tocar el código del repo.
# run_config los lee de los mismos lugares, así que quedan registrados en la metadata.json de cada modelo.
OVERRIDES = (
    "import runpy, sys; "
    "import src.constants as c; "
    f"c.CLASS_WEIGHT_POWER = {CLASS_WEIGHT_POWER}; "
    "from src.problem import PunctuationProblem; "
    f"PunctuationProblem.bidirectional_rnn_weight_decay = {PUNCTUATION_BIRNN_WEIGHT_DECAY}; "
    "sys.argv = [\"src.pipeline\"] + sys.argv[1:]; "
    "runpy.run_module(\"src.pipeline\", run_name=\"__main__\")"
)
PIPELINE = f"MPLBACKEND=Agg uv run python -c '{OVERRIDES}'"

# Mejor F1 macro existente sobre CONTROL por problema y modelo (promedio de las tareas del problema).
#   21-24-10: corrida del repo (~62 mil instancias de train, 5 épocas)
#   13-37-08: corrida local (5000 por fuente, 15 épocas)
BASELINE_CONTROL = {
    ("CapitalizationProblem", "bidirectional_rnn"): (0.6047, "21-24-10"),
    ("CapitalizationProblem", "joint_conv_bilstm"): (0.6163, "13-37-08"),
    ("PunctuationProblem", "bidirectional_rnn"): (0.6675, "21-24-10"),  # ini 0.789, fin 0.546
    ("PunctuationProblem", "joint_conv_bilstm"): (0.6685, "13-37-08"),  # ini 0.754, fin 0.583
}
RUNS: dict[str, str | None] = {}  # corrida de cada experimento

import json


def run_reports() -> dict[str, dict]:
    """Reporte de cada corrida, armado con la metadata.json de sus modelos (models/runs/<Problema>/<score>__<modelo>/)."""
    reports = {}
    for path in sorted(Path("models/runs").glob("*/*/metadata.json")):
        meta = json.loads(path.read_text())
        report = reports.setdefault(meta["run"], {"config": meta["config"], "problems": {}})
        report["problems"].setdefault(meta["problem"], {})[meta["model"]] = meta
    return reports


## 3. Entrenamiento

Una celda por modelo: si Colab corta la sesión, lo que ya terminó queda guardado en Drive (`MyDrive/tp2-models/<fecha>__<hora>/`).

### `bidirectional_rnn`

In [ ]:
before = set(run_reports())
!{PIPELINE} --experiments bidirectional_rnn --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}
new_runs = sorted(set(run_reports()) - before)
RUNS["bidirectional_rnn"] = new_runs[-1] if new_runs else None
print("Corrida:", RUNS["bidirectional_rnn"])

### `joint_conv_bilstm`

In [ ]:
before = set(run_reports())
!{PIPELINE} --experiments joint_conv_bilstm --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}
new_runs = sorted(set(run_reports()) - before)
RUNS["joint_conv_bilstm"] = new_runs[-1] if new_runs else None
print("Corrida:", RUNS["joint_conv_bilstm"])

## 4. Resultados

F1 macro sobre **control** contra el mejor resultado existente de cada modelo. Si `Δ` es positivo, el candidato mejora. Si se reinició el runtime, completar `RUNS` a mano con las corridas (campo `run` de la metadata.json de cada modelo), ej: `RUNS = {"bidirectional_rnn": "2026-11-18__10-00-00"}`.

In [ ]:
def model_rows(report: dict, run: str) -> list[tuple[str, str, int, dict[str, float], dict[str, float]]]:
    """(problema, modelo, mejor época, F1 test por tarea, F1 control por tarea) de una corrida."""
    print(run, "config:", {k: report["config"][k] for k in
          ("n_epochs", "class_weight_power", "bidirectional_rnn_weight_decay")})
    rows = []
    for problem, models in report["problems"].items():
        for model, r in models.items():
            test = {t: v["f1_macro"] for t, v in r["test"].items()}
            control = {t: v["f1_macro"] for t, v in (r.get("control") or {}).items()}
            rows.append((problem, model, r["best_epoch"], test, control))
    return rows


def mean(scores: dict[str, float]) -> float:
    """Promedio de las tareas de un problema (el score del pipeline)."""
    return sum(scores.values()) / len(scores) if scores else float("nan")


header = f"{'problema':22} {'modelo':18} {'época':>5} {'control por tarea':34} {'control':>7} {'mejor previo':>13} {'Δ':>7}"
print(header)
reports = run_reports()
for experiment, run in RUNS.items():
    if run is None or run not in reports:
        print(f"{experiment}: sin modelos guardados (¿se cortó la corrida?)")
        continue
    for problem, model, epoch, test, control in model_rows(reports[run], run):
        base, source = BASELINE_CONTROL.get((problem, model), (float("nan"), "-"))
        tasks = " ".join(f"{t}={v:.3f}" for t, v in control.items())
        delta = mean(control) - base
        print(f"{problem:22} {model:18} {epoch:5} {tasks:34} {mean(control):7.3f} {base:7.3f} ({source}) {delta:+7.3f}")